# HMR2.0 (4D-Humans) on Fit3D frames with and without the disc

Runs the 3D-body-prior model on the same frames as the RTMPose/MediaPipe experiment (clean and disc over the knee) and saves its
2D joint positions, so `src/fit3d/eval_hmr.py` can compare how much the hip angle changes.
Needs a GPU runtime (Runtime > Change runtime type > T4 GPU) and, from you: the SMPL neutral model file and `fit3d_hmr.zip`.
The person box comes from the true joints (in the zip), so no detector is installed.
Fit3D must not be redistributed: use the zip only inside your own Colab session.

In [ ]:
import torch, sys
print(torch.__version__, torch.cuda.is_available(), sys.version)
assert torch.cuda.is_available(), "switch the runtime to a GPU"

## 1. Install 4D-Humans (the step most likely to need fixing; paste any error back)

In [ ]:
!git clone https://github.com/shubham-goel/4D-Humans.git
%cd 4D-Humans
!pip install -q -e . 
!pip install -q git+https://github.com/mattloper/chumpy

## 2. SMPL neutral model
Register at https://smplify.is.tue.mpg.de/ , download the SMPLify package and get `basicModel_neutral_lbs_10_207_0_v1.0.0.pkl`
(the 4D-Humans README asks for it in `./data/`). Upload it here.

In [ ]:
from google.colab import files
import os, shutil
up = files.upload()   # choose basicModel_neutral_lbs_10_207_0_v1.0.0.pkl
name = list(up)[0]
os.makedirs('data', exist_ok=True)
shutil.copy(name, 'data/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl')
cache = os.path.expanduser('~/.cache/4DHumans/data/smpl')
os.makedirs(cache, exist_ok=True)
shutil.copy(name, os.path.join(cache, 'SMPL_NEUTRAL.pkl'))   # second location some versions look in
print(os.listdir('data'), os.listdir(cache))

## 3. Upload the frames (`outputs/fit3d_hmr.zip` from your repo folder)

In [ ]:
up = files.upload()   # choose fit3d_hmr.zip
import zipfile
zipfile.ZipFile(list(up)[0]).extractall('/content/frames')
print(len(os.listdir('/content/frames')), 'entries')

## 4. Load the model

In [ ]:
import numpy as np, cv2, json, torch
from pathlib import Path
from hmr2.models import download_models, load_hmr2, DEFAULT_CHECKPOINT
from hmr2.utils import recursive_to
from hmr2.datasets.vitdet_dataset import ViTDetDataset

download_models()
model, model_cfg = load_hmr2(DEFAULT_CHECKPOINT)
model = model.cuda().eval()
print('loaded')

## 5. Run on every frame, clean and hidden, and save the 2D joints

In [ ]:
def cam_crop_to_full(cam_bbox, box_center, box_size, img_size, focal_length):
    img_w, img_h = img_size[:, 0], img_size[:, 1]
    cx, cy, b = box_center[:, 0], box_center[:, 1], box_size
    bs = b * cam_bbox[:, 0] + 1e-9
    tz = 2 * focal_length / bs
    tx = (2 * (cx - img_w / 2.) / bs) + cam_bbox[:, 1]
    ty = (2 * (cy - img_h / 2.) / bs) + cam_bbox[:, 2]
    return torch.stack([tx, ty, tz], dim=-1)

def joints_2d(path, box):
    img = cv2.imread(path)
    ds = ViTDetDataset(model_cfg, img, np.array([box], dtype=np.float32))
    batch = recursive_to(next(iter(torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False))), 'cuda')
    with torch.no_grad():
        out = model(batch)
    isz = batch['img_size'].float()
    f = model_cfg.EXTRA.FOCAL_LENGTH / model_cfg.MODEL.IMAGE_SIZE * isz.max()
    t = cam_crop_to_full(out['pred_cam'], batch['box_center'].float(), batch['box_size'].float(), isz, f)
    X = out['pred_keypoints_3d'][0] + t[0]                      # (44, 3), camera frame
    uv = f * X[:, :2] / X[:, 2:3] + isz[0] / 2
    return uv.cpu().numpy()

boxes = json.load(open('/content/frames/boxes.json'))
names = sorted(boxes)
clean, hidden = [], []
for k, n in enumerate(names):
    subj, fr = n.split('/')
    clean.append(joints_2d(f'/content/frames/{subj}/clean_{fr}.jpg', boxes[n]))
    hidden.append(joints_2d(f'/content/frames/{subj}/hidden_{fr}.jpg', boxes[n]))
    if k % 50 == 0:
        print(k, '/', len(names), flush=True)
np.savez('hmr2_keypoints.npz', names=np.array(names), clean=np.array(clean), hidden=np.array(hidden))
print('saved', np.array(clean).shape)
files.download('hmr2_keypoints.npz')

Save the downloaded `hmr2_keypoints.npz` into `outputs/fit3d_hmr/` in the repo, then run `python src/fit3d/eval_hmr.py`.